# Colored completion demo — real PoinTr geometry + color propagation (Colab, GPU)

A quick end-to-end **colored point-cloud completion** demo:

1. Take a colored GT cloud, remove a region -> **colored partial** (PoinTr ShapeNet-55 crop logic).
2. Feed its **xyz** to the pretrained **PoinTr (ShapeNet-55)** -> completed **geometry**.
3. **Propagate color**: each completed point copies RGB from its nearest partial point.
4. Visualize partial / completion / GT + report Chamfer (geometry) and ΔE (color).

> Honest scope: PoinTr completes **geometry only**. Color here comes from a trivial
> nearest-neighbor propagation (this is the "geometry model + trivial color head"
> baseline). A real colored model would learn color; that needs training.

**Prereqs:** run your PoinTr Colab setup first (repo at `/content/PoinTr`, CUDA extensions
built, GPU runtime). Then run these cells top to bottom.

In [ ]:
# ============================================================
# Cell A - Locate PoinTr, install demo deps
# ============================================================
import os, sys, subprocess
POINTR = "/content/PoinTr"
assert os.path.isdir(POINTR), "PoinTr not found at /content/PoinTr - run your setup notebook first."
os.chdir(POINTR)
subprocess.run("pip install -q gdown open3d plotly", shell=True, check=True)
import torch
print("cwd:", os.getcwd(), "| CUDA:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# ensure PoinTr CUDA extensions are built (some setups skip emd, which metrics.py needs)
for ext in ["emd", "chamfer_dist", "cubic_feature_sampling"]:
    try:
        __import__({"emd":"emd","chamfer_dist":"chamfer","cubic_feature_sampling":"cubic_feature_sampling"}[ext])
    except Exception:
        print("building", ext, "...")
        subprocess.run(f"cd {POINTR}/extensions/{ext} && python setup.py install",
                       shell=True, check=False)


In [ ]:
# ============================================================
# Cell B - Download the pretrained ShapeNet-55 PoinTr checkpoint (robust)
# ------------------------------------------------------------
# gdown (Google Drive) is more reliable than the Tsinghua wget, which can silently
# save an HTML page. We then ASSERT the size so a bad download fails loudly here
# (a tiny "checkpoint" -> random weights -> blob completion).
# ============================================================
import os, subprocess
os.makedirs("ckpts", exist_ok=True)
CKPT = "ckpts/PoinTr_ShapeNet55.pth"
if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 50e6:
    subprocess.run(f"gdown 1WzERLlbSwzGOBybzkjBrApwyVMTG00CJ -O {CKPT}", shell=True, check=True)
    # fallback: wget -O ckpts/PoinTr_ShapeNet55.pth "https://cloud.tsinghua.edu.cn/f/4a7027b83da343bb9ac9/?dl=1"
sz = os.path.getsize(CKPT) / 1e6
assert sz > 50, f"checkpoint too small ({sz:.1f} MB) - download failed, try the wget fallback"
print(f"checkpoint OK: {sz:.1f} MB")


In [ ]:
# ============================================================
# Cell C - Upload the colored GT and make a colored partial
# ------------------------------------------------------------
# Upload data/_s2_preview/demo_gt.ply (the 8192-pt colored airplane GT).
# ============================================================
import numpy as np, open3d as o3d
from google.colab import files

up = files.upload()                       # pick demo_gt.ply
GT_PLY = list(up.keys())[0]
pc = o3d.io.read_point_cloud(GT_PLY)
gt = np.concatenate([np.asarray(pc.points), np.asarray(pc.colors)], axis=1).astype(np.float32)
print("GT:", gt.shape, "has_color:", pc.has_colors())

def separate_point_cloud(points, crop_ratio=0.5, seed=42):
    """PoinTr seprate_point_cloud (colored): remove the crop_ratio points nearest a
    random unit-vector viewpoint; keep the rest as the partial. Color rides along."""
    xyz = points[:, :3]; N = len(points); num_crop = int(round(N * crop_ratio))
    c = xyz.mean(0); xyzn = (xyz - c) / (np.linalg.norm(xyz - c, axis=1).max() + 1e-12)
    rng = np.random.default_rng(seed); v = rng.standard_normal(3); center = v / np.linalg.norm(v)
    idx = np.argsort(np.linalg.norm(xyzn - center[None], axis=1))
    return points[idx[num_crop:]], points[idx[:num_crop]]   # partial, missing

partial, missing = separate_point_cloud(gt, crop_ratio=0.5, seed=42)
print("partial:", partial.shape)

# save partial xyz (only) for PoinTr, inside an input dir
os.makedirs("/content/demo/in", exist_ok=True); os.makedirs("/content/demo/out", exist_ok=True)
pin = o3d.geometry.PointCloud(); pin.points = o3d.utility.Vector3dVector(partial[:, :3].astype(np.float64))
o3d.io.write_point_cloud("/content/demo/in/partial.ply", pin)

In [ ]:
# ============================================================
# Cell D - Run PoinTr inference (geometry completion)
# ------------------------------------------------------------
# PoinTr has a LOCAL `datasets/` package that collides with HuggingFace `datasets`.
# Prepend PoinTr to PYTHONPATH so its local package wins the import.
# ============================================================
import subprocess, os, numpy as np
env = os.environ.copy()
env["PYTHONPATH"] = "/content/PoinTr:" + env.get("PYTHONPATH", "")
cmd = ("python tools/inference.py "
       "cfgs/ShapeNet55_models/PoinTr.yaml ckpts/PoinTr_ShapeNet55.pth "
       "--pc_root /content/demo/in --out_pc_root /content/demo/out --device cuda:0")
r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="/content/PoinTr", env=env)
print(r.stdout[-2000:]); print("STDERR:", r.stderr[-2000:])
completed = np.load("/content/demo/out/partial/fine.npy")   # (M,3) completed geometry
print("completed:", completed.shape)


In [ ]:
# ============================================================
# Cell E - Propagate color + metrics (Chamfer, color ΔE)
# ============================================================
import numpy as np
from scipy.spatial import cKDTree

# color propagation: each completed point takes the RGB of its nearest partial point
ptree = cKDTree(partial[:, :3])
_, pidx = ptree.query(completed, k=1)
completed_rgb = partial[pidx, 3:6]
completed_full = np.concatenate([completed, completed_rgb], axis=1)

def srgb_to_lab(rgb):
    rgb = np.clip(rgb, 0, 1); lin = np.where(rgb > 0.04045, ((rgb + 0.055)/1.055)**2.4, rgb/12.92)
    M = np.array([[0.4124,0.3576,0.1805],[0.2126,0.7152,0.0722],[0.0193,0.1192,0.9505]])
    xyz = (lin @ M.T) / np.array([0.95047,1.0,1.08883]); d = 6/29
    f = np.where(xyz > d**3, np.cbrt(xyz), xyz/(3*d**2)+4/29)
    return np.stack([116*f[:,1]-16, 500*(f[:,0]-f[:,1]), 200*(f[:,1]-f[:,2])],1)

# geometry: symmetric Chamfer (L2) vs GT
gt_xyz = gt[:, :3]
d1, gidx = cKDTree(gt_xyz).query(completed, k=1)
d2, _ = cKDTree(completed).query(gt_xyz, k=1)
chamfer = float((d1**2).mean() + (d2**2).mean())
# color: ΔE of each completed point vs its nearest GT point
color_dE = float(np.linalg.norm(srgb_to_lab(completed_rgb) - srgb_to_lab(gt[gidx, 3:6]), axis=1).mean())
print(f"Chamfer (L2, sym): {chamfer:.6f}")
print(f"color ΔE (vs GT):  {color_dE:.3f}")

In [ ]:
# ============================================================
# Cell F - Visualize: partial / completion / GT (colored)
# ============================================================
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def sub(a, n=5000):
    return a if len(a) <= n else a[np.random.default_rng(0).choice(len(a), n, replace=False)]

def trace(arr):
    a = sub(arr)
    col = ["rgb(%d,%d,%d)" % (int(r*255), int(g*255), int(b*255)) for r, g, b in a[:, 3:6]]
    return go.Scatter3d(x=a[:,0], y=a[:,1], z=a[:,2], mode="markers", marker=dict(size=1.6, color=col))

fig = make_subplots(rows=1, cols=3, specs=[[{"type":"scene"}]*3],
                    subplot_titles=("partial (input)", "completion (PoinTr + color prop)", "GT"))
fig.add_trace(trace(partial), 1, 1)
fig.add_trace(trace(completed_full), 1, 2)
fig.add_trace(trace(gt), 1, 3)
fig.update_layout(height=500, showlegend=False, margin=dict(l=0,r=0,t=30,b=0))
for s in ("scene","scene2","scene3"): fig.layout[s].aspectmode = "data"
fig.show()